In [ ]:
from google.cloud import vision
from matplotlib import pyplot as plt
import io,os
from pdf2image import convert_from_path
from PIL import Image
import glob
 
# Set the path to your service account JSON file
service_account_json = r'C:\Users\trpwkycaudit3\End_2_End_OCR_Through_S3_Bucket\ppsl-trpw-kyc-review-4a4116f0c5c7.json'

# Instantiates a client
client = vision.ImageAnnotatorClient.from_service_account_json(service_account_json)
 
print(client)

In [ ]:
output_folder = r'C:\Users\trpwkycaudit3\End_2_End_OCR_Through_S3_Bucket\AOA_MOA_CODE\IMAGE_OUTPUT_FOLDER'
text_folder = r'C:\Users\trpwkycaudit3\End_2_End_OCR_Through_S3_Bucket\AOA_MOA_CODE\AOA_TEXT_FOLDER'
#pdf_path = r'C:\Users\trpwkycaudit3\Desktop\Aadhar_Card'text_folder

## Extract text from image

In [ ]:
def extract_text_from_image(image_path):
    try:
        with io.open(image_path, 'rb') as image_file:
            content = image_file.read()
    
        image = vision.Image(content=content)
        # Performs text detection on the image file
        response = client.text_detection(image=image)
        texts = response.text_annotations
        texts = texts[0].description

        if "MEMORANDUM OF ASSOCIATION" in texts.upper() or "ARTICLES OF ASSOCIATION" in texts.upper():
            return texts
        
    except Exception as e:
        print(e)

## converting PDF to Image first page

In [ ]:
def pdf_to_image(pdf_path,output_folder,text_folder):
    poppler_path = r'C:\poppler-24.02.0\Library\bin'
    try:    
        if os.path.exists(pdf_path) and os.path.getsize(pdf_path)>0:
            print(pdf_path)

            folder_name = os.path.splitext(os.path.basename(os.path.dirname(pdf_path)))[0]
            #folder_name = os.path.splitext(os.path.basename(pdf_path))[0]

            folder_path = os.path.join(output_folder,folder_name)
            print(folder_path)

            os.makedirs(folder_path,exist_ok=True)
            
            images = convert_from_path(pdf_path=pdf_path,first_page=0,last_page=1,poppler_path=poppler_path,dpi=500,fmt='jpeg')
    
            if images:

                file_name = os.path.splitext(os.path.basename(folder_path))[0]

                #Save jpeg file inside the created folder
                jpeg_path = os.path.join(folder_path,f"{file_name}.jpeg")
                
               
                #saving first page of image to directory
                images[0].save(jpeg_path,'JPEG')
                
                extracted_text = extract_text_from_image(jpeg_path) #this fuction used to get text from image        

                if extracted_text is not None:

                    #subfolder path to save text result
                    txt_folder_path = os.path.join(text_folder,folder_name)
    
                    os.makedirs(txt_folder_path,exist_ok=True)
                    
                    txt_file_name = os.path.splitext(os.path.basename(txt_folder_path))[0]
                    
                    text_path = os.path.join(txt_folder_path,f"{txt_file_name}.txt")
        
                    
                
                    #now saving the text to specific folder
                    with open(text_path,'w',encoding='utf-8') as file:
                     file.write(extracted_text)
                    print(f"Text File Successfully Save: {text_path}")
        
                    #now removing the image file from folder :-
                    #os.remove(output_path1)
                    #print(f"Image File removed Successfully: {output_path1}")
                else:
                    print(f"file contain None in it: {pdf_path}")
                
            else:
                print(f'No image is Found {pdf_path}')

        else:
            print(f"pdf file is either empty or doesnt exist: {pdf_path}")

    except Exception as e:
        print(e) 

In [ ]:
#pdf_to_image(pdf_path,output_folder,text_folder)

In [ ]:
main_dir = r'C:\Users\trpwkycaudit3\End_2_End_OCR_Through_S3_Bucket\AOA_MOA_CODE\AOA_MOA_DOC'
lst_pdf = []
for folder in os.listdir(main_dir):
    try:
        #print(folder)
        folder_path = os.path.join(main_dir,folder)
        #print(folder_path)
        if os.path.isdir(folder_path):
            for filename in os.listdir(folder_path):
                if filename.endswith('.pdf'):
                    full_path = os.path.join(folder_path,filename)
                    lst_pdf.append(full_path)
                    
    except Exception as e:
        print(e)
lst_pdf

In [ ]:
#getting lst of pdf file from specific directory
#lst_pdf = glob.glob(pdf_path+'/*.pdf')
try:
    for pdf_path in lst_pdf:
        pdf_to_image(pdf_path ,output_folder,text_folder)
        print("="*90)
except Exception as e:
    print(e)

In [ ]:
# path = r'C:\Users\trpwkycaudit3\End_2_End_OCR_Through_S3_Bucket\AOA_MOA_CODE\AOA_MOA_DOC\3DLogi09576162837217\DM4467266705201.pdf'
# os.path.dirname(path)
# os.path.splitext(os.path.basename(os.path.dirname(path)))[0]